In [1]:
import os
import json
import hashlib
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from umap import UMAP
import hdbscan
from anytree import Node, RenderTree

warnings.filterwarnings("ignore")

In [2]:
df_train = pd.read_json("icecat_data_train.json")
df_train.head(5)

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,ProductSeries.Language,ProductSeries.Value,BulletPoints.BulletPointsId,BulletPoints.Language,BulletPoints.Updated,BulletPoints.Values,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names
1072689,ASUS,,https://images.icecat.biz/img/brand/thumb/161_...,ASUS,https://images.icecat.biz/img/brand/thumb/161_...,K31CD-IT049T,[],153,EN,PCs/Workstations,...,None,None,NaN,None,None,None,"[{'VirtualCategoryID': 195, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...
906402,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,686915-A41,[],2509,EN,Notebook Spare Parts,...,None,None,NaN,None,None,None,None,NaN,2833>150>8355>2509,Computers & Electronics>Computers>Notebook Par...
411281,C2G,,https://images.icecat.biz/img/brand/thumb/2834...,C2G,https://images.icecat.biz/img/brand/thumb/2834...,37745,[],953,EN,Fibre Optic Cables,...,None,None,NaN,None,None,None,None,NaN,2833>830>953,Computers & Electronics>Computer Cables>Fibre ...
425903,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,FA889AA#AC3,[],8194,EN,Handheld Mobile Computer Spare Parts,...,None,None,NaN,None,None,None,None,NaN,2833>150>8194,Computers & Electronics>Computers>Handheld Mob...
1047582,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,109559U,[],153,EN,PCs/Workstations,...,None,None,NaN,None,None,None,"[{'VirtualCategoryID': 194, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...


In [3]:
df_val = pd.read_json("icecat_data_validate.json")
df_val.head(5)

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,ProductSeries.Language,ProductSeries.Value,BulletPoints.BulletPointsId,BulletPoints.Language,BulletPoints.Updated,BulletPoints.Values,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names
309544,Fujitsu,,https://images.icecat.biz/img/brand/thumb/15_b...,Fujitsu,https://images.icecat.biz/img/brand/thumb/15_b...,S26361-F5524-L800,[],1563,EN,Internal Solid State Drives,...,None,None,NaN,None,None,None,None,NaN,2833>206>2840>1563,Computers & Electronics>Data Storage>Data Stor...
805366,PanzerGlass,,https://images.icecat.biz/img/brand/thumb/1016...,PanzerGlass,https://images.icecat.biz/img/brand/thumb/1016...,PG1501,[],1568,EN,Screen Protectors,...,None,None,NaN,None,None,None,None,NaN,2833>107>1568,Computers & Electronics>Telecom & Navigation>S...
126809,2-Power,,https://images.icecat.biz/img/brand/thumb/1520...,2-Power,https://images.icecat.biz/img/brand/thumb/1520...,ALT268563B,[],911,EN,Memory Modules,...,None,None,NaN,None,None,None,None,NaN,2833>106>2844>911,Computers & Electronics>Computer Components>Sy...
922232,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,25205062,[],2509,EN,Notebook Spare Parts,...,None,None,NaN,None,None,None,None,NaN,2833>150>8355>2509,Computers & Electronics>Computers>Notebook Par...
567207,Verbatim,,https://images.icecat.biz/img/brand/thumb/669_...,Verbatim,https://images.icecat.biz/img/brand/thumb/669_...,97537,[],194,EN,Keyboards,...,None,None,NaN,None,None,None,None,NaN,2833>191>194,Computers & Electronics>Data Input Devices>Key...


In [4]:
df_test = pd.read_json("icecat_data_test.json")
df_test.head(5)

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,ProductSeries.Language,ProductSeries.Value,BulletPoints.BulletPointsId,BulletPoints.Language,BulletPoints.Updated,BulletPoints.Values,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names
500081,Fujitsu,,https://images.icecat.biz/img/brand/thumb/15_b...,Fujitsu,https://images.icecat.biz/img/brand/thumb/15_b...,FSP:G-SW3Z560PRE0S,[],788,EN,Warranty & Support Extensions,...,None,None,NaN,None,None,None,None,NaN,2833>839>788,Computers & Electronics>Warranty & Support>War...
741063,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,5KM83PA,None,151,EN,Notebooks,...,EN,13,1268560.0,EN,2018-10-21 21:55:51,"[Windows 10 Home 64-bit, Intel® Core™ i7-8565U...","[{'VirtualCategoryID': 329, 'UNCATID': '432115...",NaN,2833>150>151,Computers & Electronics>Computers>Notebooks
1091454,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,4NG33EA,[],153,EN,PCs/Workstations,...,EN,880,NaN,None,None,None,"[{'VirtualCategoryID': 195, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...
522928,C2G,,https://images.icecat.biz/img/brand/thumb/2834...,C2G,https://images.icecat.biz/img/brand/thumb/2834...,83061,[],883,EN,Networking Cables,...,None,None,NaN,None,None,None,None,NaN,2833>830>883,Computers & Electronics>Computer Cables>Networ...
479478,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,5PS0A14091,[],788,EN,Warranty & Support Extensions,...,None,None,NaN,None,None,None,None,NaN,2833>839>788,Computers & Electronics>Warranty & Support>War...


In [5]:
import pandas as pd
import re

TEXT_COLS = [
    "Brand",
    "ProductName",
    "Title",
    "Description.LongProductName",
    "Description.LongDesc",
    "SummaryDescription.LongSummaryDescription",
    "SummaryDescription.ShortSummaryDescription",
    "Category.Name.Value",
    "pathlist_names",
]

def to_text(x):
    if isinstance(x, list):
        x = " ".join(map(str, x))
    if x is None:
        return ""
    x = str(x)
    if x.lower() in ["none", "nan"]:
        return ""
    return x

def build_metadata_text(row):
    parts = [to_text(row.get(col, "")) for col in TEXT_COLS]
    return " ".join(p for p in parts if p)

def clean_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"<[^>]+>", " ", s)              # remove HTML
    s = re.sub(r"[^a-z0-9\-+x/ ]+", " ", s)     # keep limited chars
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [6]:
# take only 1000 rows to work with
df_train = df_train.sample(n=1000, random_state=42).reset_index(drop=True)

#  build metadata_text and metadata_text_clean
df_train["metadata_text"] = df_train.apply(build_metadata_text, axis=1)
df_train["metadata_text_clean"] = df_train["metadata_text"].apply(clean_text)

df_train[["metadata_text", "metadata_text_clean"]].head(2)

,metadata_text,metadata_text_clean
0,Acer 60.SH7N2.001 Acer 60.SH7N2.001 notebook s...,acer 60 sh7n2 001 acer 60 sh7n2 001 notebook s...
1,Tripp Lite Minicom Smart 108 Tripp Lite Minic...,tripp lite minicom smart 108 tripp lite minico...


In [7]:
df_val = df_val.sample(n=1000, random_state=42).reset_index(drop=True)

df_val["metadata_text"] = df_val.apply(build_metadata_text, axis=1)
df_val["metadata_text_clean"] = df_val["metadata_text"].apply(clean_text)

df_val[["metadata_text", "metadata_text_clean"]].head(2)

,metadata_text,metadata_text_clean
0,Lenovo 41Y8342 Lenovo 41Y8342 internal solid s...,lenovo 41y8342 lenovo 41y8342 internal solid s...
1,"HP 15-bs019ni HP 15-bs019ni Red,Black Notebook...",hp 15-bs019ni hp 15-bs019ni red black notebook...


In [8]:
df_test = df_test.sample(n=1000, random_state=42).reset_index(drop=True)

df_test["metadata_text"] = df_test.apply(build_metadata_text, axis=1)
df_test["metadata_text_clean"] = df_test["metadata_text"].apply(clean_text)

df_test[["metadata_text", "metadata_text_clean"]].head(2)

,metadata_text,metadata_text_clean
0,DELL 312-0106 DELL 312-0106 notebook spare par...,dell 312-0106 dell 312-0106 notebook spare par...
1,Xerox PHASER 6250DP ZW-KL LSR 24PPM 256MB 500V...,xerox phaser 6250dp zw-kl lsr 24ppm 256mb 500v...


In [9]:
def make_ft_df(df):
    df_ft = df[["metadata_text_clean", "pathlist_names"]].copy()
    df_ft = df_ft.dropna(subset=["pathlist_names"])
    df_ft = df_ft[df_ft["metadata_text_clean"] != ""]
    df_ft = df_ft.rename(columns={
        "metadata_text_clean": "input_text",
        "pathlist_names": "target_path",
    })
    return df_ft.reset_index(drop=True)

df_train_ft = make_ft_df(df_train)
df_val_ft   = make_ft_df(df_val)
df_test_ft  = make_ft_df(df_test)

df_train_ft.head(3)

,input_text,target_path
0,acer 60 sh7n2 001 acer 60 sh7n2 001 notebook s...,Computers & Electronics>Computers>Notebook Par...
1,tripp lite minicom smart 108 tripp lite minico...,Computers & Electronics>Data Input Devices>KVM...
2,hp 826630-a41 hp 826630-a41 notebook spare par...,Computers & Electronics>Computers>Notebook Par...


In [10]:
df_val_ft.head(3)

,input_text,target_path
0,lenovo 41y8342 lenovo 41y8342 internal solid s...,Computers & Electronics>Data Storage>Data Stor...
1,hp 15-bs019ni hp 15-bs019ni red black notebook...,Computers & Electronics>Computers>Notebooks
2,gigabyte geforce 8400 256mb ddr2 gigabyte gefo...,Computers & Electronics>Computer Components>Sy...


In [11]:
df_test_ft.head(3)

,input_text,target_path
0,dell 312-0106 dell 312-0106 notebook spare par...,Computers & Electronics>Computers>Notebook Par...
1,xerox phaser 6250dp zw-kl lsr 24ppm 256mb 500v...,Computers & Electronics>Printers & Scanners>Pr...
2,startech com 5 ft cat 6 white molded rj45 utp ...,Computers & Electronics>Computer Cables>Networ...


In [12]:
import json
from pathlib import Path

# New clean output folder under /home/jovyan
out_dir = Path("fine_tuning_data")   # this will be /home/jovyan/fine_tuning_data
out_dir.mkdir(parents=True, exist_ok=True)

def make_row(row):
    return {
        "messages": [
            {
                "role": "system",
                "content": "You are an assistant that assigns taxonomy paths to products.",
            },
            {
                "role": "user",
                "content": row["input_text"],
            },
            {
                "role": "assistant",
                "content": row["target_path"],
            },
        ]
    }

for name, df_ft in [("train", df_train_ft), ("val", df_val_ft), ("test", df_test_ft)]:
    out_path = out_dir / f"icecat_{name}_ft.jsonl"
    with out_path.open("w", encoding="utf-8") as f:
        for _, r in df_ft.iterrows():
            f.write(json.dumps(make_row(r), ensure_ascii=False) + "\n")
    print(f"Saved {len(df_ft)} rows to {out_path}")


Saved 1000 rows to fine_tuning_data/icecat_train_ft.jsonl
Saved 1000 rows to fine_tuning_data/icecat_val_ft.jsonl
Saved 1000 rows to fine_tuning_data/icecat_test_ft.jsonl


In [13]:
file_path = "fine_tuning_data/icecat_train_ft.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline())


{"messages": [{"role": "system", "content": "You are an assistant that assigns taxonomy paths to products."}, {"role": "user", "content": "acer 60 sh7n2 001 acer 60 sh7n2 001 notebook spare part top case top case grey acer 60 sh7n2 001 type top case brand compatibility acer compatibility chromebook c710 acer 60 sh7n2 001 top case acer chromebook c710 notebook spare parts computers electronics computers notebook parts accessories notebook spare parts"}, {"role": "assistant", "content": "Computers & Electronics>Computers>Notebook Parts & Accessories>Notebook Spare Parts"}]}

{"messages": [{"role": "system", "content": "You are an assistant that assigns taxonomy paths to products."}, {"role": "user", "content": "tripp lite minicom smart 108 tripp lite minicom smart 108 kvm switch rack mounting black 100m vga ps/2 x 2 1600 x 1200 cat5 save space and money the smart 108 is a single-user analog cat5 kvm switch that gives you the ability to control multiple computers or servers from a single 

In [14]:
import os
os.getcwd()


'/home/jovyan'

In [15]:
#Load your JSONL into a HuggingFace Dataset

In [16]:
from datasets import load_dataset
import torch

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

train_path = "fine_tuning_data/icecat_train_ft.jsonl"
val_path   = "fine_tuning_data/icecat_val_ft.jsonl"
test_path  = "fine_tuning_data/icecat_test_ft.jsonl"

train_ds = load_dataset("json", data_files=train_path)["train"]
val_ds   = load_dataset("json", data_files=val_path)["train"]
test_ds  = load_dataset("json", data_files=test_path)["train"]

print(train_ds[0])


Device: cuda


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

{'messages': [{'role': 'system', 'content': 'You are an assistant that assigns taxonomy paths to products.'}, {'role': 'user', 'content': 'acer 60 sh7n2 001 acer 60 sh7n2 001 notebook spare part top case top case grey acer 60 sh7n2 001 type top case brand compatibility acer compatibility chromebook c710 acer 60 sh7n2 001 top case acer chromebook c710 notebook spare parts computers electronics computers notebook parts accessories notebook spare parts'}, {'role': 'assistant', 'content': 'Computers & Electronics>Computers>Notebook Parts & Accessories>Notebook Spare Parts'}]}


In [17]:
# Convert messages → text format for LLaMA ( convert chat format to llama format so that llama understands the data better)

In [18]:
def messages_to_text(example):
    s = example["messages"][0]["content"]
    u = example["messages"][1]["content"]
    a = example["messages"][2]["content"]

    return {
        "text": f"<s>[INST] {s}\n{u} [/INST] {a}</s>"
    }

train_ds = train_ds.map(messages_to_text)
val_ds   = val_ds.map(messages_to_text)
test_ds  = test_ds.map(messages_to_text)


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [19]:
# we plug in LLaMA-3.1-8B-Instruct (local) + LoRA and train.
# This is the QLoRA magic:
# load_in_4bit=True → loads model in 4-bit compressed weights
# → reduces GPU memory from 16GB → ~3GB
# nf4 → best quantization format for LLMs
# double quantization → further reduces memory
# compute in BF16 → fast + stable training
# Without QLoRA you cannot train an 8B model on a single GPU.

# Without LoRA:
# You would try to update all 8 billion parameters of LLaMA → IMPOSSIBLE on a normal GPU.
# With LoRA:
# You freeze the 8B original model and train only a very small number of extra parameters.
# Example:
# Model	Total Params	Trainable Params	GPU Memory Needed
# Full LLaMA-3 8B	8,000,000,000	8,000,000,000	❌ 45–70 GB (impossible)
# QLoRA (LLaMA-3 8B)	8,000,000,000	~4,000,000	✅ 10–16 GB

In [20]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model
import torch

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

# use your local Llama model
model_name = "models/meta-llama/Llama-3.1-8B-Instruct"   # relative to /home/jovyan

# 4-bit quantization so 8B fits on GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Loads vocab + token IDs for LLaMA
# Adds a pad token (LLaMA doesn't have one)
# Sets padding direction to "right"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

# LoRA adapter (we only train a small part of the model)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


Device: cuda


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848


In [21]:
# Tokenize datasets for LLaMA

In [22]:
MAX_LEN = 256  

def tokenize_fn(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )
    # labels = input_ids (causal LM learns to predict whole sequence)
    enc["labels"] = enc["input_ids"].copy()
    return enc

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(tokenize_fn,   batched=True, remove_columns=val_ds.column_names)

train_tok.set_format(type="torch")
val_tok.set_format(type="torch")

print(train_tok[0].keys())


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [23]:
from accelerate import Accelerator

# Save original method
_orig_unwrap_model = Accelerator.unwrap_model

def _patched_unwrap_model(self, model, *args, **kwargs):
    # Drop unexpected kwarg if present
    kwargs.pop("keep_torch_compile", None)
    return _orig_unwrap_model(self, model, *args, **kwargs)

Accelerator.unwrap_model = _patched_unwrap_model

print("✅ Patched Accelerator.unwrap_model to ignore keep_torch_compile")


✅ Patched Accelerator.unwrap_model to ignore keep_torch_compile


In [24]:
# TrainingArguments + Trainer + train

In [26]:
output_dir = "llama_taxonomy_1000"

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=10,           #  will print training loss every 10 steps
    logging_first_step=True,
    report_to="none",
)


In [27]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
)

train_result = trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("LLaMA fine-tuning on 1000 samples finished.")


Step,Training Loss
1,3.245100
10,3.109900
20,2.035400
30,1.445000
40,1.592100
50,1.447400
60,1.388200
70,1.479400
80,1.315300
90,1.130900


LLaMA fine-tuning on 1000 samples finished.


In [28]:
# final train loss
print("Train loss:", train_result.training_loss)

# run validation explicitly
eval_metrics = trainer.evaluate()
print("Eval metrics:", eval_metrics)        # contains 'eval_loss'
print("Validation loss:", eval_metrics["eval_loss"])


Train loss: 1.5789303550720215


Eval metrics: {'eval_loss': 1.339426040649414, 'eval_runtime': 117.0706, 'eval_samples_per_second': 8.542, 'eval_steps_per_second': 4.271, 'epoch': 1.0}
Validation loss: 1.339426040649414


In [ ]:
# Load fine-tuned LLaMA & evaluate on test set

In [29]:
import torch; torch.cuda.empty_cache()

In [30]:

# this part is only using the test data — we’re doing it to:
# Measure how good Approach 2 really is (accuracy on unseen products)
# Show clean GOLD vs PRED examples for your thesis
# Debug formatting so the model outputs nice A>B>C paths

In [31]:
#---on test data

In [35]:
# ============================================
# 1. Imports & paths
# ============================================
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import re

model_path = "llama_taxonomy_1000"                    # your fine-tuned model
test_path  = "fine_tuning_data/icecat_test_ft.jsonl"  # test JSONL


# ============================================
# 2. Load test dataset (messages format)
# ============================================
test_ds = load_dataset("json", data_files=test_path)["train"]
print("Test sample 0:", test_ds[0])


# ============================================
# 3. Load fine-tuned LLaMA model + tokenizer
# ============================================
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

device = 0 if torch.cuda.is_available() else -1
print("Device:", "cuda" if device == 0 else "cpu")

gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device,
)


# ============================================
# 4. Prompt builder (for test + user input)
# ============================================
SYSTEM_PROMPT = (
    "You are an assistant that assigns taxonomy paths to products.\n"
    "Return ONLY the taxonomy path in the format:\n"
    "A > B > C\n"
    "Do NOT add explanations.\n"
    "Do NOT repeat the input.\n"
    "Do NOT include special tokens.\n"
)

def build_prompt(user_text: str) -> str:
    return f"<s>[INST] {SYSTEM_PROMPT}\nProduct description:\n{user_text}\n[/INST]"


# ============================================
# 5. Clean taxonomy extractor (IMPROVED)
# ============================================
def extract_clean_taxonomy(text: str) -> str:
    """
    Extract the first clean A>B>C style taxonomy from raw model output.
    """

    # keep only text after [/INST]
    if "[/INST]" in text:
        text = text.split("[/INST]", 1)[1]

    # remove special tokens
    text = (text.replace("<s>", "")
                .replace("</s>", "")
                .replace("[INST]", "")
                .replace("[/INST]", "")
                .strip())

    # regex for path-like structure
    pattern = r"[A-Za-z0-9&/ ,\-]+>(?:[A-Za-z0-9&/ ,\-]+>)*[A-Za-z0-9&/ ,\-]+"
    match = re.search(pattern, text)

    if match:
        return match.group(0).strip()

    return ""


# ============================================
# 6. Normalize paths for comparison
# ============================================
def normalize_path(s: str) -> str:
    return s.replace(" ", "").strip().lower()


# ============================================
# 7. Evaluation on test set
# ============================================
def eval_on_test(ds, max_examples=100, max_new_tokens=50):
    n = min(max_examples, len(ds))
    correct = 0

    for i in range(n):
        ex = ds[i]

        user_text = ex["messages"][1]["content"]
        gold = ex["messages"][2]["content"].strip()

        prompt = build_prompt(user_text)

        out = gen(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )[0]["generated_text"]

        pred = extract_clean_taxonomy(out).strip()

        if i < 5:
            print(f"\n--- Example {i} ---")
            print("USER TEXT (truncated):", user_text[:200], "...")
            print("GOLD:", gold)
            print("PRED:", pred)

        if normalize_path(pred) == normalize_path(gold):
            correct += 1

    acc = correct / n * 100
    print(f"\nExact-match accuracy on first {n} test items: {acc:.2f}% ({correct}/{n})")


# Run evaluation
eval_on_test(test_ds, max_examples=100, max_new_tokens=50)


# ============================================
# 8. Interactive single prediction (user input)
# ============================================
def predict_taxonomy():
    print("\nEnter product description (Ctrl+C to stop):\n")
    user_text = input("> ")

    prompt = build_prompt(user_text)

    out = gen(
        prompt,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )[0]["generated_text"]

    pred = extract_clean_taxonomy(out).strip()

    print("\n=== Predicted Taxonomy Path ===")
    print(pred or "(model did not output a clear taxonomy)")
    print("================================")


# Uncomment to test manually:
# predict_taxonomy()


Test sample 0: {'messages': [{'role': 'system', 'content': 'You are an assistant that assigns taxonomy paths to products.'}, {'role': 'user', 'content': 'dell 312-0106 dell 312-0106 notebook spare part battery 1900 mah 14 8 v always on the go- no more worries for running out of battery power you can back up your system with this 4-cell lithium-ion primary battery from dell it has an internal circuit board with chips that allow it to communicate with the notebook to monitor battery performance output voltage and temperature it also gives the notebook an accurate fuel gauge capability to determine how much battery runtime is left before the next recharge is required this product has been tested and validated on dell systems to ensure it will work with your computer and is compatible with dell latitude x300 notebook it is supported by dell technical support when used with a dell system highlights 28 whr capacity lets you work seamlessly while on the move provides very safe charging and di

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


Device: cuda

--- Example 0 ---
USER TEXT (truncated): dell 312-0106 dell 312-0106 notebook spare part battery 1900 mah 14 8 v always on the go- no more worries for running out of battery power you can back up your system with this 4-cell lithium-ion prim ...
GOLD: Computers & Electronics>Computers>Notebook Parts & Accessories>Notebook Spare Parts
PRED: Computers & Electronics>Computers>Notebook Parts & Accessories>Notebook Spare Parts

--- Example 1 ---
USER TEXT (truncated): xerox phaser 6250dp zw-kl lsr 24ppm 256mb 500vl+100 dupl 2400dpi eth xerox phaser 6250dp zw-kl lsr 24ppm 256mb 500vl+100 dupl 2400dpi eth colour 600 x 600 dpi a4 phaser 6250dp a4/legal size color prin ...
GOLD: Computers & Electronics>Printers & Scanners>Printing Equipment>Laser Printers
PRED: Computers & Electronics>Printers & Scanners>Printing Equipment>Laser Printers

--- Example 2 ---
USER TEXT (truncated): startech com 5 ft cat 6 white molded rj45 utp gigabit cat6 patch cable - 5ft patch cord startech com 5 

In [38]:
# ============================================
# 1. Imports & paths
# ============================================
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import re

model_path = "llama_taxonomy_1000"                    # your fine-tuned model
test_path  = "fine_tuning_data/icecat_test_ft.jsonl"  # test JSONL


# ============================================
# 2. Load test dataset (messages format)
# ============================================
test_ds = load_dataset("json", data_files=test_path)["train"]
print("Test sample 0:", test_ds[0])


# ============================================
# 3. Load fine-tuned LLaMA model + tokenizer
# ============================================
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

device = 0 if torch.cuda.is_available() else -1
print("Device:", "cuda" if device == 0 else "cpu")

gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device,
)


# ============================================
# 4. Prompt builder (for test + user input)
# ============================================
SYSTEM_PROMPT = (
    "You are an assistant that assigns taxonomy paths to products.\n"
    "Return ONLY the taxonomy path in the format:\n"
    "A > B > C\n"
    "Do NOT add explanations.\n"
    "Do NOT repeat the input.\n"
    "Do NOT include special tokens.\n"
)

def build_prompt(user_text: str) -> str:
    return f"<s>[INST] {SYSTEM_PROMPT}\nProduct description:\n{user_text}\n[/INST]"


# ============================================
# 5. Clean taxonomy extractor (IMPROVED)
# ============================================
# def extract_clean_taxonomy(text: str) -> str:
#     """
#     Extract the first clean A>B>C style taxonomy from raw model output.
#     """

#     # keep only text after [/INST]
#     if "[/INST]" in text:
#         text = text.split("[/INST]", 1)[1]

#     # remove special tokens
#     text = (text.replace("<s>", "")
#                 .replace("</s>", "")
#                 .replace("[INST]", "")
#                 .replace("[/INST]", "")
#                 .strip())

#     # regex for path-like structure
#     pattern = r"[A-Za-z0-9&/ ,\-]+>(?:[A-Za-z0-9&/ ,\-]+>)*[A-Za-z0-9&/ ,\-]+"
#     match = re.search(pattern, text)

#     if match:
#         return match.group(0).strip()

#     return ""
def extract_clean_taxonomy(text: str) -> str:
    """
    Extract the cleanest taxonomy path (A > B > C style) from raw model output.
    """

    # keep only text after [/INST] if present
    if "[/INST]" in text:
        text = text.split("[/INST]", 1)[1]

    # remove special tokens and common junk phrases
    junk_phrases = [
        "<s>", "</s>",
        "[INST]", "[/INST]",
        "A > B > C",
        "Product taxonomy path",
        "/INST>",
    ]
    for jp in junk_phrases:
        text = text.replace(jp, "")

    text = text.strip()

    # find ALL path-like matches and take the LAST one
    pattern = r"[A-Za-z0-9&/ ,\-]+>(?:[A-Za-z0-9&/ ,\-]+>)*[A-Za-z0-9&/ ,\-]+"
    matches = re.findall(pattern, text)

    if not matches:
        return ""

    path = matches[-1].strip()  # last match is usually the clean one

    # normalize spaces around '>'
    levels = [lvl.strip() for lvl in path.split(">")]
    path = " > ".join(levels)

    return path


# ============================================
# 6. Normalize paths for comparison
# ============================================
def normalize_path(s: str) -> str:
    return s.replace(" ", "").strip().lower()


# ============================================
# 7. Evaluation on test set
# ============================================
def eval_on_test(ds, max_examples=100, max_new_tokens=50):
    n = min(max_examples, len(ds))
    correct = 0

    for i in range(n):
        ex = ds[i]

        user_text = ex["messages"][1]["content"]
        gold = ex["messages"][2]["content"].strip()

        prompt = build_prompt(user_text)

        out = gen(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )[0]["generated_text"]

        pred = extract_clean_taxonomy(out).strip()

        if i < 5:
            print(f"\n--- Example {i} ---")
            print("USER TEXT (truncated):", user_text[:200], "...")
            print("GOLD:", gold)
            print("PRED:", pred)

        if normalize_path(pred) == normalize_path(gold):
            correct += 1

    acc = correct / n * 100
    print(f"\nExact-match accuracy on first {n} test items: {acc:.2f}% ({correct}/{n})")


# Run evaluation
eval_on_test(test_ds, max_examples=100, max_new_tokens=50)


# ============================================
# 8. Interactive single prediction (user input)
# ============================================
def predict_taxonomy():
    print("\nEnter product description (Ctrl+C to stop):\n")
    user_text = input("> ")

    prompt = build_prompt(user_text)

    out = gen(
        prompt,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )[0]["generated_text"]

    pred = extract_clean_taxonomy(out).strip()

    print("\n=== Predicted Taxonomy Path ===")
    print(pred or "(model did not output a clear taxonomy)")
    print("================================")


# Uncomment to test manually:
predict_taxonomy()


Test sample 0: {'messages': [{'role': 'system', 'content': 'You are an assistant that assigns taxonomy paths to products.'}, {'role': 'user', 'content': 'dell 312-0106 dell 312-0106 notebook spare part battery 1900 mah 14 8 v always on the go- no more worries for running out of battery power you can back up your system with this 4-cell lithium-ion primary battery from dell it has an internal circuit board with chips that allow it to communicate with the notebook to monitor battery performance output voltage and temperature it also gives the notebook an accurate fuel gauge capability to determine how much battery runtime is left before the next recharge is required this product has been tested and validated on dell systems to ensure it will work with your computer and is compatible with dell latitude x300 notebook it is supported by dell technical support when used with a dell system highlights 28 whr capacity lets you work seamlessly while on the move provides very safe charging and di

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


Device: cuda

--- Example 0 ---
USER TEXT (truncated): dell 312-0106 dell 312-0106 notebook spare part battery 1900 mah 14 8 v always on the go- no more worries for running out of battery power you can back up your system with this 4-cell lithium-ion prim ...
GOLD: Computers & Electronics>Computers>Notebook Parts & Accessories>Notebook Spare Parts
PRED: Computers & Electronics > Computers > Notebook Parts & Accessories > Notebook Spare Parts

--- Example 1 ---
USER TEXT (truncated): xerox phaser 6250dp zw-kl lsr 24ppm 256mb 500vl+100 dupl 2400dpi eth xerox phaser 6250dp zw-kl lsr 24ppm 256mb 500vl+100 dupl 2400dpi eth colour 600 x 600 dpi a4 phaser 6250dp a4/legal size color prin ...
GOLD: Computers & Electronics>Printers & Scanners>Printing Equipment>Laser Printers
PRED: Computers & Electronics > Printers & Scanners > Printing Equipment > Laser Printers

--- Example 2 ---
USER TEXT (truncated): startech com 5 ft cat 6 white molded rj45 utp gigabit cat6 patch cable - 5ft patch cord sta

>  startech com 5 ft cat 6 white molded rj45 utp gigabit cat6 patch cable - 5ft patch cord startech com 5 ft cat 6 white molded rj45 utp gigabit cat6 patch cable - 5ft patch cord startech com 5 ft cat 6  ... GOLD: Computers & Electronics>Computer Cables>Networking Cables



=== Predicted Taxonomy Path ===
Computers & Electronics > Computer Cables > Networking Cables


In [42]:
import textwrap

def print_wrong_predictions_only(ds, max_examples=100, max_new_tokens=50):
    """
    Print ONLY wrong predictions.
    """
    n = min(max_examples, len(ds))
    wrong_cases = []

    print(f"Scanning first {n} test samples...\n")

    for i in range(n):
        ex = ds[i]

        # extract text + gold
        user_text = ex["messages"][1]["content"]
        gold = ex["messages"][2]["content"].strip()

        # build prompt
        prompt = build_prompt(user_text)

        # prediction
        out = gen(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )[0]["generated_text"]

        pred = extract_clean_taxonomy(out).strip()

        # store only wrong results
        if pred != gold:
            wrong_cases.append((i, user_text, gold, pred))

    # print results
    print("=" * 60)
    print(f" WRONG CASES FOUND: {len(wrong_cases)} / {n}")
    print("=" * 60)

    for (idx, user_text, gold, pred) in wrong_cases:
        print(f"\n--- WRONG CASE {idx} ---")
        print("USER TEXT:", textwrap.shorten(user_text, width=250, placeholder=" ..."))
        print("\nGOLD:", gold)
        print("PRED:", pred)
        print("-" * 60)

    print(f"\nTotal wrong predictions: {len(wrong_cases)} / {n}\n")


In [43]:
print_wrong_predictions_only(test_ds, max_examples=100, max_new_tokens=50)


Scanning first 100 test samples...

 WRONG CASES FOUND: 100 / 100

--- WRONG CASE 0 ---
USER TEXT: dell 312-0106 dell 312-0106 notebook spare part battery 1900 mah 14 8 v always on the go- no more worries for running out of battery power you can back up your system with this 4-cell lithium-ion primary battery from dell it has an internal ...

GOLD: Computers & Electronics>Computers>Notebook Parts & Accessories>Notebook Spare Parts
PRED: Computers & Electronics > Computers > Notebook Parts & Accessories > Notebook Spare Parts
------------------------------------------------------------

--- WRONG CASE 1 ---
USER TEXT: xerox phaser 6250dp zw-kl lsr 24ppm 256mb 500vl+100 dupl 2400dpi eth xerox phaser 6250dp zw-kl lsr 24ppm 256mb 500vl+100 dupl 2400dpi eth colour 600 x 600 dpi a4 phaser 6250dp a4/legal size color printer 220v 26ppm color/b w 24ppm a4 color/b w ...

GOLD: Computers & Electronics>Printers & Scanners>Printing Equipment>Laser Printers
PRED: Computers & Electronics > Printers &

In [44]:
import re
import textwrap

# -------------------------
# 1. Normalize a taxonomy path into clean segments
# -------------------------
def normalize_path(path: str):
    """
    Turn a taxonomy string into a normalized list of segments.
    - Lowercase
    - Remove extra spaces
    - Replace '&' and '/' with spaces so 'Warranty/Support' ~= 'Warranty & Support'
    """
    if not path:
        return []

    # remove extra special tokens just in case
    path = path.replace("<s>", "").replace("</s>", "")
    path = path.replace("[INST]", "").replace("[/INST]", "")
    path = path.strip()

    # split on '>'
    raw_segs = path.split(">")

    clean_segs = []
    for seg in raw_segs:
        seg = seg.strip().lower()
        if not seg:
            continue
        # unify & and / into spaces
        seg = seg.replace("&", " ")
        seg = seg.replace("/", " ")
        # collapse multiple spaces
        seg = re.sub(r"\s+", " ", seg).strip()
        clean_segs.append(seg)

    return clean_segs


# -------------------------
# 2. Relaxed equality between GOLD and PRED
# -------------------------
def relaxed_equal(gold: str, pred: str) -> bool:
    """
    Relaxed rule:
    - Ignore case and spacing around '>'
    - Treat 'warranty/support' and 'warranty & support' as equal
    - Allow PRED to miss the leading 'computers electronics' root
    """
    g = normalize_path(gold)
    p = normalize_path(pred)

    if not g or not p:
        return False

    # exact normalized match
    if g == p:
        return True

    # allow missing root "computers electronics" in PRED
    # e.g. GOLD: ["computers electronics", "warranty support", "warranty support extensions"]
    #      PRED:               ["warranty support", "warranty support extensions"]
    if len(p) == len(g) - 1 and g[0] == "computers electronics" and g[1:] == p:
        return True

    return False


# -------------------------
# 3. Level-wise comparison (A / B / C)
# -------------------------
def level_wise_match(gold: str, pred: str):
    """
    Compare level-wise:
    level 0 -> A, level 1 -> B, level 2 -> C, etc.
    Uses normalized segments.
    """
    g = normalize_path(gold)
    p = normalize_path(pred)

    # pad shorter list with None for safe indexing
    max_len = max(len(g), len(p))
    g += [None] * (max_len - len(g))
    p += [None] * (max_len - len(p))

    # return booleans for first 3 levels (A,B,C)
    matches = []
    for lvl in range(3):
        if lvl >= max_len or g[lvl] is None or p[lvl] is None:
            matches.append(None)  # level not defined
        else:
            matches.append(g[lvl] == p[lvl])
    return matches  # [A_match, B_match, C_match]


In [45]:
def eval_relaxed_with_levels(ds, max_examples=100, max_new_tokens=50):
    n = min(max_examples, len(ds))

    strict_correct = 0
    relaxed_correct = 0

    # level-wise stats
    level_total = [0, 0, 0]     # A, B, C defined in gold
    level_correct = [0, 0, 0]   # A, B, C correct

    for i in range(n):
        ex = ds[i]
        user_text = ex["messages"][1]["content"]
        gold = ex["messages"][2]["content"].strip()

        prompt = build_prompt(user_text)

        out = gen(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )[0]["generated_text"]

        pred = extract_clean_taxonomy(out).strip()

        # --- print first few for sanity ---
        if i < 5:
            print(f"\n--- Example {i} ---")
            print("USER TEXT (truncated):", textwrap.shorten(user_text, width=200, placeholder=" ..."))
            print("GOLD:", gold)
            print("PRED:", pred)

        # 1) strict exact string equality
        if pred == gold:
            strict_correct += 1

        # 2) relaxed equality
        if relaxed_equal(gold, pred):
            relaxed_correct += 1

        # 3) level-wise
        lvl_matches = level_wise_match(gold, pred)
        for lvl in range(3):
            if lvl_matches[lvl] is None:
                continue
            level_total[lvl] += 1
            if lvl_matches[lvl]:
                level_correct[lvl] += 1

    strict_acc = strict_correct / n * 100
    relaxed_acc = relaxed_correct / n * 100

    print(f"\nStrict exact-match accuracy on first {n}: {strict_acc:.2f}% ({strict_correct}/{n})")
    print(f"Relaxed accuracy (normalized, root-tolerant): {relaxed_acc:.2f}% ({relaxed_correct}/{n})")

    # level-wise
    labels = ["A-level", "B-level", "C-level"]
    for lvl in range(3):
        if level_total[lvl] == 0:
            continue
        acc_lvl = level_correct[lvl] / level_total[lvl] * 100
        print(f"{labels[lvl]} correct: {acc_lvl:.2f}% ({level_correct[lvl]}/{level_total[lvl]})")


In [46]:
eval_relaxed_with_levels(test_ds, max_examples=100, max_new_tokens=50)



--- Example 0 ---
USER TEXT (truncated): dell 312-0106 dell 312-0106 notebook spare part battery 1900 mah 14 8 v always on the go- no more worries for running out of battery power you can back up your system with this 4-cell lithium-ion ...
GOLD: Computers & Electronics>Computers>Notebook Parts & Accessories>Notebook Spare Parts
PRED: Computers & Electronics > Computers > Notebook Parts & Accessories > Notebook Spare Parts

--- Example 1 ---
USER TEXT (truncated): xerox phaser 6250dp zw-kl lsr 24ppm 256mb 500vl+100 dupl 2400dpi eth xerox phaser 6250dp zw-kl lsr 24ppm 256mb 500vl+100 dupl 2400dpi eth colour 600 x 600 dpi a4 phaser 6250dp a4/legal size color ...
GOLD: Computers & Electronics>Printers & Scanners>Printing Equipment>Laser Printers
PRED: Computers & Electronics > Printers & Scanners > Printing Equipment > Laser Printers

--- Example 2 ---
USER TEXT (truncated): startech com 5 ft cat 6 white molded rj45 utp gigabit cat6 patch cable - 5ft patch cord startech com 5 ft cat 6 wh